In [2]:
import pathlib
import numpy as np
import scipy.sparse as sp
import scipy.io
import pandas as pd
import networkx as nx
import utils.preprocess
from sklearn.model_selection import train_test_split

save_prefix = r'./data\yelp2_processed\graph_split/'

num_ntypes = 5

In [3]:
dat = pd.read_csv(r'./data/yelp/yelp_triple.dat', sep = ' ', header=None)
business_category = pd.read_csv(r'./data/yelp/yelp_business_category.txt', sep = '\t', header=None)

In [4]:
dat.head(10)

,0,1,2
0,0,2614,0
1,2614,0,1
2,0,2652,0
3,2652,0,1
4,0,2667,0
5,2667,0,1
6,0,2707,0
7,2707,0,1
8,0,2714,0
9,2714,0,1


In [6]:
np.unique(dat[0].values).shape

(3913,)

In [7]:
dat.shape

(77360, 3)

In [8]:
edge0 = dat.iloc[dat.values[:,2] == 0]
edge0.shape #(30838, 3)
edge1 = dat.iloc[dat.values[:,2] == 1]
edge1.shape #(30838, 3)
edge2 = dat.iloc[dat.values[:,2] == 2]
edge2.shape #(2614, 3)
edge3 = dat.iloc[dat.values[:,2] == 3]
edge3.shape #(2614, 3)
edge4 = dat.iloc[dat.values[:,2] == 4]
edge4.shape #(2614, 3)
edge5 = dat.iloc[dat.values[:,2] == 5]
edge5.shape #(2614, 3)
edge6 = dat.iloc[dat.values[:,2] == 6]
edge6.shape #(2614, 3)
edge7 = dat.iloc[dat.values[:,2] == 7]
edge7.shape #(2614, 3)

(2614, 3)

In [9]:
tmp = [min(dat.iloc[dat.values[:,2] == p][0]) for p in np.unique(dat[2].values)]
tmp

[0, 2614, 0, 3900, 0, 3911, 0, 3902]

In [10]:
tmp2 = [max(dat.iloc[dat.values[:,2] == p][0]) for p in np.unique(dat[2].values)]
tmp2

[2613, 3899, 2613, 3901, 2613, 3912, 2613, 3910]

In [11]:
tmp3 = [np.unique(dat[dat[2] == p][0]).shape for p in np.unique(dat[2].values)]
tmp3

[(2614,), (1286,), (2614,), (2,), (2614,), (2,), (2614,), (9,)]

In [12]:
edge5.head(3)

,0,1,2
66905,3911,267,5
66907,3911,1328,5
66909,3911,698,5


## type_mask

In [13]:
raw_dims = [2614, 1286, 2, 9, 2]
dim = np.unique(dat[0].values).shape[0] # 3913
print(raw_dims)
print(dim)


prefix_operator = np.ones((num_ntypes+1, num_ntypes))
prefix_operator = np.tril(prefix_operator, k=-1)   # 下三角矩阵
prefix_operator = prefix_operator.dot(raw_dims).astype(int)
print(prefix_operator)

# 0 for business 2614, 1 for user 1286, 2 for service 2, 3 for star level 9, 4 for reservation 2
type_mask = np.zeros(dim,dtype=int)
for i in range(num_ntypes):
    type_mask[prefix_operator[i]:prefix_operator[i+1]] = i

np.save(save_prefix + 'prefix_operator.npy',prefix_operator)
np.save(save_prefix + 'type_mask.npy',type_mask)

[2614, 1286, 2, 9, 2]
3913
[   0 2614 3900 3902 3911 3913]


## adjM

In [14]:
adjM = sp.lil_matrix((dim,dim)) # 3913
for head,tail,_ in dat.values:
    adjM[head,tail] = 1

scipy.sparse.save_npz(save_prefix + 'adjM.npz', adjM.tocsr())


## labels

In [15]:
business_category.head(5)

,0,1
0,267,1
1,1328,1
2,698,1
3,2211,2
4,274,2


In [16]:
[business_category.shape, min(business_category[0]), max(business_category[0])]

[(2614, 2), 0, 2613]

In [17]:
x,y = np.unique(business_category[1],return_counts=True)
[x,y]

[array([0, 1, 2], dtype=int64), array([ 470, 1082, 1062], dtype=int64)]

In [18]:
business_category = business_category.sort_values(by=[0],ascending=True)
business_category.values

array([[   0,    1],
       [   1,    0],
       [   2,    0],
       ...,
       [2611,    1],
       [2612,    1],
       [2613,    1]], dtype=int64)

## ontology

In [19]:
prefix_operator = np.load(save_prefix + 'prefix_operator.npy')
type_mask = np.load(save_prefix + 'type_mask.npy')
adjM = scipy.sparse.load_npz(save_prefix + 'adjM.npz')
ontology = {
    'stem':[1,0,3],
    'branch':{0:[0,2],1:[0,4]}
}
ontology_pairs = [[0,1],[0,2],[0,3],[0,4]]

In [20]:
link_intances = utils.preprocess.get_intances_by_pair(adjM, type_mask, ontology, prefix_operator)
#print(link_intances)
print('=======done=======')

# nodes 3913
# stem 30838
# branch 2614, 2614

=======done=======


In [21]:
link_intances.keys()

dict_keys(['stem', 'branch'])

In [22]:
link_intances['branch'][0].shape

(2614, 2)

In [23]:
ontology = {
    'stem':[1,0,3],
    'branch':{0:[0,2],1:[0,4]}
}
subgraphs = utils.preprocess.get_ontology_subgraphs_v3(ontology, link_intances)

subgraphs = subgraphs[subgraphs.columns.sort_values()]
print(len(subgraphs))
print(subgraphs)

# 30838 subgraphs

30838
        0     1     2     3     4
0     177  3144  3901  3905  3911
1     177  3147  3901  3905  3911
2     177  3153  3901  3905  3911
3     177  3163  3901  3905  3911
4     177  3178  3901  3905  3911
..    ...   ...   ...   ...   ...
333  2480  3701  3901  3906  3911
334  2571  2886  3900  3902  3912
335  2571  2946  3900  3902  3912
336  2547  2944  3900  3905  3912
337  2510  3033  3901  3902  3911

[30838 rows x 5 columns]


In [24]:
import time

In [25]:
ontology = {
    'stem':[0,1],
    'branch':{0:[0,2],1:[0,4],2:[0,3]}
}


In [26]:
t = time.time()
link_intances = utils.preprocess.get_intances_by_pair(adjM, type_mask, ontology, prefix_operator)
subgraphs = utils.preprocess.get_ontology_subgraphs_v2(ontology, link_intances)
print(time.time()-t)
subgraphs = subgraphs[subgraphs.columns.sort_values()]
print(len(subgraphs))
print(subgraphs)

1.331355094909668
30838
          0     1     2     3     4
0         0  2614  3900  3906  3912
1         0  2652  3900  3906  3912
2         0  2667  3900  3906  3912
3         0  2707  3900  3906  3912
4         0  2714  3900  3906  3912
...     ...   ...   ...   ...   ...
30833  2609  3353  3900  3907  3911
30834  2610  2861  3900  3908  3912
30835  2611  3353  3900  3905  3912
30836  2612  3353  3900  3905  3912
30837  2613  2798  3900  3902  3912

[30838 rows x 5 columns]


In [27]:
subgraphs.loc[subgraphs.iloc[:,1]==2614].shape

(13, 5)

In [28]:
subgraphs.loc[subgraphs.iloc[:,0]==1]

,0,1,2,3,4
37,1,2615,3900,3905,3911
38,1,2823,3900,3905,3911
39,1,2837,3900,3905,3911
40,1,3140,3900,3905,3911
41,1,3175,3900,3905,3911
42,1,3249,3900,3905,3911
43,1,3349,3900,3905,3911
44,1,3356,3900,3905,3911
45,1,3762,3900,3905,3911


In [29]:
subgraphs.columns

Index([0, 1, 2, 3, 4], dtype='object')

## search incomplete

In [30]:
ontology_pairs = [[0,1],[0,2],[0,3],[0,4]]
res_adj = utils.preprocess.find_res_adj2(adjM, subgraphs)

In [31]:
incomplete_ontology_subgraphs, incomplete_subgraphs = utils.preprocess.find_incomplete_subgraph(adjM, type_mask, ontology_pairs, res_adj)
print(len(incomplete_ontology_subgraphs))
print(incomplete_subgraphs)
# 10s

Thu Apr 18 10:42:47 2024, finding pairs...
Thu Apr 18 10:42:47 2024, finding pairs...
Thu Apr 18 10:42:47 2024, finding pairs...
Thu Apr 18 10:42:47 2024, finding pairs...
0
[]


## save

In [32]:
# create the directories if they do not exist
for i in ['complete','incomplete']:
    pathlib.Path(save_prefix + '{}'.format(i)).mkdir(parents=True, exist_ok=True)

# save data 

# mapping of node to subgraphs

# mapping of node to node pairs 

# save schema
np.save(save_prefix + 'complete/ontology.npy', ontology) # schema
np.save(save_prefix + 'ontology_pairs.npy', ontology_pairs)
# subgraph table
np.save(save_prefix + 'complete/subgraphs.npy', subgraphs)
# all nodes adjacency matrix
scipy.sparse.save_npz(save_prefix + 'adjM.npz', adjM)
# all nodes features one-hot
for i in range(num_ntypes):
    scipy.sparse.save_npz(save_prefix + 'features_{}.npz'.format(i), scipy.sparse.eye(raw_dims[i]).tocsr())
# all nodes type labels
np.save(save_prefix + 'node_types.npy', type_mask)
# type prefix
np.save(save_prefix + 'prefix_operator.npy', prefix_operator)
# business labels
np.save(save_prefix + 'labels.npy', business_category.values) # 3 class

## split

In [33]:
# subgraphs train/validation/test splits
rand_seed = 123456789
train_val_idx, test_idx = train_test_split(np.arange(adjM.shape[0]), test_size=0.11, random_state=rand_seed)
a = np.isin(subgraphs,test_idx)
a = a.sum(axis=1).astype('bool')
subgraphs_test = subgraphs[a]
subgraphs_tr_val = subgraphs[~a]
subgraphs[a].shape
print(subgraphs_test.shape[0]/len(subgraphs)) # 30% for test
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.08, random_state=rand_seed)
b = np.isin(subgraphs_tr_val,val_idx)
b = b.sum(axis=1).astype('bool')
subgraphs_val = subgraphs_tr_val[b]
subgraphs_train = subgraphs_tr_val[~b]
subgraphs_tr_val[b].shape
print(subgraphs_val.shape[0]/len(subgraphs)) # 10% for val
print(len(subgraphs_train)/len(subgraphs)) # 60% for train


np.savez(save_prefix + 'complete/' + 'subgraphs_train_val_test.npz',
         subgraphs_train=subgraphs_train,
         subgraphs_val=subgraphs_val,
         subgraphs_test=subgraphs_test) # subgraph train&val&test
# node split
np.savez(save_prefix + 'complete/' + 'train_val_test_nodes.npz',
         train_nodes=train_idx,
         val_nodes=val_idx,
         test_nodes=test_idx) # nodes train&val&test

0.3182112977495298
0.08473312147350671
0.5970555807769635


In [43]:
adjM[np.unique(test_idx)].sum()/adjM.sum()

0.11458117890382627

In [44]:
adjM[np.unique(val_idx)].sum()/adjM.sum()

0.05443381592554292